In [44]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
import time

In [45]:
class PipelineState(TypedDict):
    step1_result: str
    step2_result: str
    step3_result: str
    fail_step2: bool

In [46]:
def step1(state: PipelineState) -> dict:
    print("Running Step 1...")
    time.sleep(1)

    return {
        "step1_result": "Data fetched successfully"
    }
    
def step2(state: PipelineState) -> dict:
    print("Running Step 2...")
    time.sleep(1)

    if state.get("fail_step2"):
        print("Step 2 CRASHED!")

        raise Exception("Network/API error")

    return {
        "step2_result": "Data processed successfully"
    }
    
def step3(state: PipelineState) -> dict:
    print("Running Step 3...")
    time.sleep(1)

    return {
        "step3_result": "Data saved successfully"
    }

In [47]:
builder = StateGraph(PipelineState)

builder.add_node("step1", step1)
builder.add_node("step2", step2)
builder.add_node("step3", step3)

builder.add_edge(START, "step1")
builder.add_edge("step1", "step2")
builder.add_edge("step2", "step3")
builder.add_edge("step3", END)

In [48]:
checkpointer = MemorySaver()

graph = builder.compile(
    checkpointer=checkpointer
)

config = {
    "configurable": {
        "thread_id": "pipeline-1"
    }
}

In [49]:
try:
    graph.invoke(
        {
            "fail_step2": True
        },
        config=config,
        durability="sync"
    )

except Exception as e:
    print("\nPipeline failed:", e)

Running Step 1...
Running Step 2...
Step 2 CRASHED!

Pipeline failed: Network/API error


In [50]:
state = graph.get_state(config)

print("\n--- SAVED STATE ---")
print(state.values)

print("\n--- NEXT NODE ---")
print(state.next)


--- SAVED STATE ---
{'step1_result': 'Data fetched successfully', 'fail_step2': True}

--- NEXT NODE ---
('step2',)


In [51]:
graph.update_state(
    config,
    {
        "fail_step2": False
    }
)

print("Failure fixed!")

Failure fixed!


In [52]:
result = graph.invoke(
    None,
    config=config
)

print("\n--- FINAL RESULT ---")
print(result)

Running Step 2...
Running Step 3...

--- FINAL RESULT ---
{'step1_result': 'Data fetched successfully', 'step2_result': 'Data processed successfully', 'step3_result': 'Data saved successfully', 'fail_step2': False}
